In [23]:
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv
load_dotenv()


True

In [24]:
model = ChatGroq(
            model="llama-3.1-8b-instant",
            temperature=0.0,
            max_retries=2,
        
        )

In [25]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    evaluation: str

In [26]:
def create_outline(state: BlogState) -> BlogState:
    
    #fetch the title from the state
    title = state['title']
    
    #call the model to create an outline for the blog post
    prompt = f"Generate an detailed outline for a blog on the topic: {title}"
    outline = model.invoke(prompt).content
    
    # update the state with the outline and return it
    state['outline'] = outline
    return state

In [27]:
def create_blog(state: BlogState) -> BlogState:
    
    #fetch the title and outline from the state
    title = state['title']
    outline = state['outline']
    
    #call the model to create a blog post based on the title and outline
    prompt = f"Write a detailed blog post based on the following title and outline:\n\nTitle: {title}\n\nOutline: {outline}"
    content = model.invoke(prompt).content
    
    # update the state with the content and return it
    state['content'] = content
    return state

In [28]:
def evaluate_blog(state: BlogState) -> BlogState:
    #based on the content of the blog post, evaluate its quality and return the state with an added evaluation field
    content = state['content']
     
    #call the model to evaluate the blog post content
    prompt = f"Evaluate the quality of the following blog post content evaluate an integer score from 1 to 10:\n{content}"
    evaluation = model.invoke(prompt).content
    
    #update the state with the evaluation and return it
    state['evaluation'] = evaluation
    return state

In [29]:
graph = StateGraph(BlogState)

#nodes 
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)
graph.add_node('evaluate_blog',evaluate_blog) #for simplicity, we will use the same function to evaluate the blog post

#edges
graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog') 
graph.add_edge('create_blog','evaluate_blog') 
graph.add_edge('evaluate_blog',END)

#compile the graph

workflow = graph.compile()



In [31]:
initial_state = BlogState(title="The Future of AI in Healthcare", outline="", content="")

final_state = workflow.invoke(initial_state)

print(f"Content*********: {final_state['content']}")
print(f"Evaluation*********: {final_state['evaluation']}")

Content*********: **The Future of AI in Healthcare: Revolutionizing Patient Care and Outcomes**

The healthcare industry is on the cusp of a revolution, driven by the rapid advancement of artificial intelligence (AI) technologies. AI has the potential to transform the way healthcare is delivered, from diagnosis and treatment to patient engagement and outcomes. However, the future of AI in healthcare is uncertain, and its benefits and challenges must be carefully considered.

**Current Applications of AI in Healthcare**

AI has already made significant inroads in healthcare, with various applications across different domains. Some of the most notable examples include:

1. **Medical Imaging Analysis**: AI-powered algorithms can analyze medical images, such as X-rays, CT scans, and MRIs, to detect abnormalities and diagnose diseases more accurately and quickly than human radiologists.
2. **Clinical Decision Support Systems (CDSSs)**: AI-powered CDSSs can provide healthcare professionals w